<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_24_iterators_advanced/note_lesson_24_iterators_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 24 — Ітератори advanced

Диспетчерська «Смачно + Таксі» отримує потік подій від кур'єрів: `picked`, `delivered`, серед них биті рядки й події попередньої зміни. Сьогодні:

1. **Клас-ітератор** з `__iter__` і `__next__`, ітерабельне проти ітератора.
2. **Генератор зсередини**: стани, `return`, `close()`, `.send()`, автомат, `yield from`.
3. **`itertools`**: `dropwhile`, `takewhile`, `groupby`, `pairwise`, `accumulate`, `combinations`.
4. **ETL-конвеєр** подій зі стадіями, які легко тестувати.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, схеми й архітектура — у книзі: [Урок 24. Ітератори advanced](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_24/).

## 🔁 Пригадай (без підглядання)

1. Що робить `for` з об'єктом перед першим елементом?
2. Чому другий `list()` від генератора порожній?
3. Що повертав `__iter__` кошика в уроці 23?

<details>
<summary>Відповіді</summary>

1. Викликає `iter()`, далі `next()` до `StopIteration`.
2. Генератор одноразовий.
3. Готовий ітератор словника.

</details>

In [ ]:
LOG = [
    "17:52 D-2 98 delivered",
    "17:58 D-1 97 delivered",
    "18:03 D-1 101 picked",
    "18:05 D-2 102 picked",
    "18:07 D-3 103 picked",
    "18:21 D-1 101 delivered",
    "18:24 ?? зламаний рядок",
    "18:29 D-2 102 delivered",
    "18:31 D-1 104 picked",
    "18:44 D-3 103 delivered",
    "18:52 D-1 104 delivered",
]
print(len(LOG))

## 1. Клас-ітератор

**Прогноз:** що надрукує другий `list(log)`?

In [ ]:
class ShiftLog:
    def __init__(self, lines):
        self.lines = lines
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.lines):
            raise StopIteration
        line = self.lines[self.position]
        self.position += 1
        return line


log = ShiftLog(LOG[2:5])
print(list(log))
print(list(log))

<details>
<summary>Відповідь</summary>

`[]`: курсор `position` живе в самому журналі й після першого проходу стоїть у кінці.

</details>

In [ ]:
class ShiftLogIterator:
    def __init__(self, lines):
        self._lines = lines
        self._position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._position >= len(self._lines):
            raise StopIteration
        line = self._lines[self._position]
        self._position += 1
        return line


class ShiftLog:
    def __init__(self, lines):
        self.lines = list(lines)

    def __iter__(self):
        return ShiftLogIterator(self.lines)


log = ShiftLog(LOG[2:5])
print(len(list(log)), len(list(log)))
print(len([(a, b) for a in log for b in log]))

In [ ]:
class ShiftLog:
    def __init__(self, lines):
        self.lines = list(lines)

    def __iter__(self):
        for line in self.lines:
            yield line


log = ShiftLog(LOG[2:5])
print(len(list(log)), len(list(log)))

## 🛠 Вправа 1. Курсор з `peek()`

`Peekable(iterable)` — ітератор, у якого `peek(default=None)` показує наступний елемент, **не забираючи** його. Має працювати і з нескінченним потоком.

In [ ]:
from itertools import count


class Peekable:
    def __init__(self, iterable):
        self._it = iter(iterable)
        self._buffer = []

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __iter__(self):
        return self

    def __next__(self):
        if self._buffer:
            return self._buffer.pop()
        return next(self._it)

    def peek(self, default=None):
        if not self._buffer:
            try:
                self._buffer.append(next(self._it))
            except StopIteration:
                return default
        return self._buffer[0]
    # END SOLUTION


cursor = Peekable(LOG[2:5])
print(cursor.peek())
assert cursor.peek() == "18:03 D-1 101 picked" == next(cursor)
assert next(cursor) == "18:05 D-2 102 picked"
assert cursor.peek() == "18:07 D-3 103 picked"
assert list(cursor) == ["18:07 D-3 103 picked"]
assert cursor.peek() is None and cursor.peek("кінець") == "кінець"
numbers = Peekable(count())
assert numbers.peek() == 0 and next(numbers) == 0 and next(numbers) == 1
print("✅ Вправа 1 пройдена")

## 2. Генератор зсередини

**Прогноз:** коли надрукується «між подіями»?

In [ ]:
from inspect import getgeneratorstate


def shift():
    print("зміна почалась")
    yield "перша подія"
    print("між подіями")
    yield "друга подія"
    print("зміна закінчилась")
    return "звіт готовий"


gen = shift()
print(getgeneratorstate(gen))
print(next(gen))
print(getgeneratorstate(gen))
print(next(gen))
try:
    next(gen)
except StopIteration as stop:
    print("StopIteration:", stop.value)
print(getgeneratorstate(gen))

<details>
<summary>Відповідь</summary>

З другим `next()`: кожен `next()` виконує тіло від попереднього `yield` до наступного. `return` потрапляє в `StopIteration.value`.

</details>

In [ ]:
def read_events(lines):
    print("відкрили з'єднання")
    try:
        for line in lines:
            yield line
    finally:
        print("закрили з'єднання")


events = read_events(LOG)
print(next(events))
events.close()
print(getgeneratorstate(events))

### `.send()`

**Прогноз:** що надрукують `next(avg)` і чотири `send`?

In [ ]:
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        minutes = yield average
        total += minutes
        count += 1
        average = round(total / count, 1)


avg = running_average()
print(next(avg))
for minutes in [18, 24, 37, 21]:
    print(avg.send(minutes))

<details>
<summary>Відповідь</summary>

`None`, а далі середнє після кожної доставки: 18.0, 21.0, 26.3, 25.0.

</details>

In [ ]:
fresh = running_average()
try:
    fresh.send(18)
except TypeError as error:
    print(error)

In [ ]:
from functools import wraps


def primed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        gen = func(*args, **kwargs)
        next(gen)
        return gen
    return wrapper


@primed
def running_average():
    total = 0
    count = 0
    average = None
    while True:
        minutes = yield average
        total += minutes
        count += 1
        average = round(total / count, 1)


avg = running_average()
print(avg.send(18), avg.send(24))

In [ ]:
COURIER_TRANSITIONS = {
    ("вільний", "picked"): "везе",
    ("везе", "delivered"): "вільний",
}


@primed
def courier_state():
    state = "вільний"
    while True:
        event = yield state
        new_state = COURIER_TRANSITIONS.get((state, event))
        if new_state is None:
            print(f"  подія {event} у стані «{state}» — пропускаю")
        else:
            state = new_state


courier = courier_state()
for event in ["picked", "delivered", "delivered", "picked"]:
    print(event, "→", courier.send(event))

## 🛠 Вправа 2. Сопрограма-лічильник доставок

`delivery_counter()` з `@primed` приймає через `send` пари `(courier, kind)` і віддає **копію** словника «кур'єр → кількість доставок». Події `picked` не рахуються.

In [ ]:
@primed
def delivery_counter():
    # YOUR CODE HERE
    # BEGIN SOLUTION
    counts = {}
    while True:
        courier, kind = yield dict(counts)
        if kind == "delivered":
            counts[courier] = counts.get(courier, 0) + 1
    # END SOLUTION


counter = delivery_counter()
print(counter.send(("D-1", "picked")))
assert counter.send(("D-1", "delivered")) == {"D-1": 1}
assert counter.send(("D-2", "delivered")) == {"D-1": 1, "D-2": 1}
snapshot = counter.send(("D-1", "delivered"))
assert snapshot == {"D-1": 2, "D-2": 1}
snapshot["D-1"] = 100
assert counter.send(("D-3", "picked")) == {"D-1": 2, "D-2": 1}, "віддавай копію, а не сам словник"
print("✅ Вправа 2 пройдена")

In [ ]:
def valid_events(lines):
    broken = 0
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            broken += 1
            continue
        yield parts
    return broken


def with_report(lines):
    broken = yield from valid_events(lines)
    print(f"битих рядків: {broken}")


events = list(with_report(LOG))
print(len(events), events[2])

## 3. `itertools`

**Прогноз:** скільки подій у першій половині зміни (18:00–18:30)?

In [ ]:
from itertools import dropwhile, takewhile

shift_events = dropwhile(lambda line: line < "18:00", LOG)
print(next(shift_events))

first_half = takewhile(lambda line: line < "18:30", dropwhile(lambda line: line < "18:00", LOG))
print(len(list(first_half)))

<details>
<summary>Відповідь</summary>

6, включно з битим рядком 18:24 — він теж «рядок до 18:30». Биті рядки відкидає розбір, а не межі часу.

</details>

**Прогноз:** скільки груп дасть `groupby` за кур'єром на несортованих подіях?

In [ ]:
from itertools import groupby

groups = [(courier, len(list(group))) for courier, group in groupby(events, key=lambda event: event[1])]
print(len(groups), groups[:4])

<details>
<summary>Відповідь</summary>

9: `groupby` групує лише сусідні елементи.

</details>

In [ ]:
by_courier = sorted(events, key=lambda event: event[1])
print([(courier, len(list(group))) for courier, group in groupby(by_courier, key=lambda event: event[1])])

In [ ]:
from itertools import accumulate, pairwise


def minutes(hhmm):
    hours, mins = hhmm.split(":")
    return int(hours) * 60 + int(mins)


times = [minutes(event[0]) for event in events]
gaps = [later - earlier for earlier, later in pairwise(times)]
print(gaps, max(gaps))

delivered = [1 if event[3] == "delivered" else 0 for event in events]
print(list(accumulate(delivered)))

In [ ]:
from itertools import combinations

fares = [230, 150, 270, 180, 120, 410]
print([pair for pair in combinations(fares, 2) if sum(pair) == 500])
print([trio for trio in combinations(fares, 3) if sum(trio) == 500])
print(sum(1 for size in range(1, len(fares) + 1) for _ in combinations(fares, size)))

## 🛠 Вправа 3. Пачки: `batched`

`batched(iterable, n)` віддає кортежі по `n`, останній може бути коротшим. Має працювати з нескінченним потоком. (У Python 3.12+ є `itertools.batched`; тут пишемо свій.)

In [ ]:
from itertools import count, islice


def batched(iterable, n):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    iterator = iter(iterable)
    while True:
        batch = tuple(islice(iterator, n))
        if not batch:
            return
        yield batch
    # END SOLUTION


print(list(batched(range(7), 3)))
assert list(batched(range(7), 3)) == [(0, 1, 2), (3, 4, 5), (6,)]
assert list(batched([], 3)) == []
assert next(batched(count(), 2)) == (0, 1)
assert [len(batch) for batch in batched(LOG, 4)] == [4, 4, 3]
print("✅ Вправа 3 пройдена")

## 🛠 Вправа 4. Ковзне середнє

`moving_average(values, k)` — середнє останніх `k` значень для кожної позиції, починаючи з `k`-ї. Не перераховуй суму вікна щоразу (ковзне вікно з уроку 11). Має працювати з будь-яким ітерабельним, зокрема генератором.

In [ ]:
from collections import deque


def moving_average(values, k):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    window = deque()
    window_sum = 0
    for value in values:
        window.append(value)
        window_sum += value
        if len(window) > k:
            window_sum -= window.popleft()
        if len(window) == k:
            yield window_sum / k
    # END SOLUTION


print(list(moving_average([18, 24, 37, 21], 2)))
assert list(moving_average([18, 24, 37, 21], 2)) == [21.0, 30.5, 29.0]
assert list(moving_average([10, 20, 30], 3)) == [20.0]
assert list(moving_average([10, 20], 3)) == []
assert list(moving_average((m for m in [4, 8, 6, 2]), 2)) == [6.0, 7.0, 4.0]
print("✅ Вправа 4 пройдена")

## 4. ETL-конвеєр: середній час доставки по кур'єрах

In [ ]:
def parse(lines, rejected):
    for line in lines:
        parts = line.split()
        if len(parts) != 4 or not parts[2].isdigit():
            rejected.append(line)
            continue
        time, courier, order, kind = parts
        yield {"time": minutes(time), "courier": courier, "order": int(order), "kind": kind}


def in_shift(events, start):
    return dropwhile(lambda event: event["time"] < start, events)


def durations(events):
    picked = {}
    for event in events:
        if event["kind"] == "picked":
            picked[event["order"]] = event["time"]
        elif event["kind"] == "delivered" and event["order"] in picked:
            yield event["courier"], event["time"] - picked.pop(event["order"])


def report(pairs):
    by_courier = {}
    for courier, spent in pairs:
        by_courier.setdefault(courier, []).append(spent)
    return {courier: round(sum(spent) / len(spent), 1) for courier, spent in sorted(by_courier.items())}


rejected = []
pipeline = durations(in_shift(parse(LOG, rejected), minutes("18:00")))
print(report(pipeline))
print(rejected)

## 🛠 Вправа 5. Підсумок зміни через `groupby`

`shift_summary(events)` приймає список подій `[час, кур'єр, замовлення, подія]` (як `events` вище) і повертає словник «кур'єр → (скільки забрав, скільки доставив)». Використай `sorted` + `groupby`.

In [ ]:
from itertools import groupby


def shift_summary(events):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    summary = {}
    ordered = sorted(events, key=lambda event: event[1])
    for courier, group in groupby(ordered, key=lambda event: event[1]):
        kinds = [event[3] for event in group]
        summary[courier] = (kinds.count("picked"), kinds.count("delivered"))
    return summary
    # END SOLUTION


print(shift_summary(events))
assert shift_summary(events) == {"D-1": (2, 3), "D-2": (1, 2), "D-3": (1, 1)}
assert shift_summary([]) == {}
print("✅ Вправа 5 пройдена")

## ✅ Самоперевірка

1. Чим ітерабельне відрізняється від ітератора? Що повертає `__iter__` кожного з них?
2. Куди потрапляє `return` генератора і хто його забирає?
3. Навіщо запускати генератор перед першим `send`?
4. Чому `takewhile` працює з нескінченним потоком, а `filter` — ні?
5. Коли `groupby` дає неправильні групи?
6. Чому кожну стадію ETL-конвеєра легко тестувати?

<details>
<summary>Відповіді</summary>

1. Ітерабельне на кожен `iter()` дає новий курсор; ітератор — сам курсор, його `__iter__` повертає `self`.
2. У `StopIteration.value`; його отримує `yield from`.
3. Щоб генератор дійшов до першого `yield`, який прийме значення.
4. `takewhile` зупиняється на першому хибному елементі; `filter` перевіряє всі.
5. Коли однакові ключі не йдуть підряд — дані треба спершу відсортувати.
6. Стадія приймає будь-яке ітерабельне: для тесту досить списку з кількох рядків.

</details>

### Шпаргалка

```python
class Cursor:                          # клас-ітератор
    def __iter__(self): return self
    def __next__(self): ...            # елемент або raise StopIteration

class Log:                             # ітерабельне: новий курсор щоразу
    def __iter__(self):
        yield from self.lines

value = yield result                   # сопрограма: send(value)
gen = coro(); next(gen); gen.send(x)   # або декоратор @primed
result = yield from sub()              # делегування + return-значення

dropwhile(pred, it)  takewhile(pred, it)  groupby(sorted(it, key=k), key=k)
pairwise(it)  accumulate(it)  combinations(it, r)  islice(it, n)
```

## Далі

- **Урок 25 — Тестування з pytest**: стадії конвеєра — перші кандидати на тести.
- **Урок 27 — asyncio**: `async` / `await` виросли з `.send()` і `yield from`.